In [ ]:
import sys
sys.path.insert(0, "..")

from typing import Literal
from pydantic import BaseModel, Field


class Fact(BaseModel):
    measure: Literal["HR", "median", "rate", "difference", "n", "events", "p", "other"] = Field(
        description="что это за число: HR; median — медиана; rate — доля или частота, %; difference — разница; "
                    "n — число пациентов; events — число событий; p — p-значение; other — другое")
    endpoint: str = Field(description="конечная точка коротко, как на слайде: PFS, OS, DFS, ORR, CR, DOR, pCR; "
                                      "для характеристик пациентов — baseline")
    group: str = Field(description="к какой группе, сравнению или подгруппе относится: 'EV+P', 'EV+P vs chemotherapy', "
                                   "'all patients', 'age ≥65'")
    value: str = Field(description="значение ровно как напечатано, без единиц: '0.58', '<0.001', '33.6', 'NR'")
    unit: str | None = Field(description="единица, если напечатана: '%', 'mo'; иначе null")
    ci_level: str | None = Field(description="уровень доверительного интервала, как напечатан: '95', '99.5'; иначе null")
    ci_low: str | None = Field(description="нижняя граница интервала, как напечатана; иначе null")
    ci_high: str | None = Field(description="верхняя граница интервала, как напечатана; иначе null")
    timepoint: str | None = Field(description="срок, если указан: '3 yr', '36 mo', '4-year'; иначе null")


class SlideFacts(BaseModel):
    title: str | None = Field(description="заголовок слайда, как напечатан")
    trial_name: str | None = Field(description="название исследования, как напечатано на слайде")
    nct: str | None = Field(description="номер NCT, если напечатан; иначе null")
    citation: str | None = Field(description="ссылка на публикацию или конференцию, как напечатана; иначе null")
    slide_type: Literal["results", "design", "baseline", "text", "other"] = Field(
        description="results — результаты; design — дизайн исследования; baseline — характеристики пациентов; "
                    "text — текст без таблиц и графиков; other — другое")
    facts: list[Fact] = Field(description="все числовые результаты со слайда; у дизайна исследования — пустой список")
    uncertain: list[str] = Field(description="что прочитано неуверенно или не читается — каждое отдельной строкой")


example = Fact(measure="HR", endpoint="IDFS", group="olaparib vs placebo", value="0.58", unit=None,
               ci_level="99.5", ci_low="0.41", ci_high="0.82", timepoint=None)
print(example)
print("бланки внутри:", list(SlideFacts.model_json_schema()["$defs"]))
print("граф у факта:", len(Fact.model_fields))

In [ ]:
import base64
import io
import json
import time
from datetime import datetime
from pathlib import Path

import anthropic
from dotenv import load_dotenv
from PIL import Image

from app.config import cost_usd, READ_MODEL

load_dotenv("../.env")
client = anthropic.Anthropic()
RUN_PAID = False   # предохранитель: True — только когда сознательно запускаешь платный прогон


def image_to_base64(img):
    """Картинку Pillow → строка base64 в формате PNG (без записи на диск)."""
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    return base64.standard_b64encode(buffer.getvalue()).decode("utf-8")


def read_facts(images, model, instruction):
    """Картинки слайда → бланк SlideFacts от модели; вернуть бланк, расход, время и причину остановки."""
    if not RUN_PAID:
        raise RuntimeError("платный вызов выключен: поставь RUN_PAID = True")
    content = []
    for img in images:
        content.append({"type": "image",
                        "source": {"type": "base64", "media_type": "image/png", "data": image_to_base64(img)}})
    content.append({"type": "text", "text": instruction})
    t0 = time.perf_counter()
    response = client.messages.parse(
        model=model,
        max_tokens=16000,
        messages=[{"role": "user", "content": content}],
        output_format=SlideFacts,
    )
    return {
        "model": model,
        "reading": response.parsed_output.model_dump() if response.parsed_output else None,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "cost_usd": cost_usd(response.usage, model),
        "seconds": round(time.perf_counter() - t0, 1),
        "stop_reason": response.stop_reason,
    }


def save_run(run, name):
    """Сохранить прогон в data/runs/<дата>_<имя>.json; вернуть путь."""
    folder = Path("../data/runs")
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}_{name}.json"
    path.write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


print("модель чтения:", READ_MODEL)

In [ ]:
INSTRUCTION_DRAFT = (
    "Это слайд доклада. Перепиши в бланк ВСЕ числовые результаты, которые напечатаны на слайде, "
    "ровно как напечатаны: каждое число — отдельным фактом, интервал — в графы ci_* того числа, к которому он относится. "
    "Ничего не додумывай и не бери из памяти: только то, что видно на слайде. "
    "Числа под кривыми «No. at risk» и оси графиков не выписывай. "
    "Если число не читается — не угадывай: пропусти его и добавь пункт в uncertain."
)

olympia = Image.open(sorted(Path("../tests/slides/real").glob("*.png"))[0]).convert("RGB")
draft = read_facts([olympia], READ_MODEL, INSTRUCTION_DRAFT)
RUN_PAID = False                                             # сразу выключить обратно
print(save_run(draft, "draft_olympia"))

print(f"${draft['cost_usd']:.4f} · {draft['seconds']} с · {draft['stop_reason']} · выход {draft['output_tokens']} токенов")
reading = draft["reading"]
print("исследование:", reading["trial_name"], "· тип:", reading["slide_type"], "· фактов:", len(reading["facts"]))
for f in reading["facts"]:
    ci = f"({f['ci_level']}% ДИ {f['ci_low']}–{f['ci_high']})" if f["ci_low"] else ""
    print(f"  {f['measure']:10} {f['endpoint']:8} {f['group'][:34]:34} {f['value']:>8} {f['unit'] or '':3} {ci}")
print("сомнения:", reading["uncertain"])

In [ ]:
gold = json.loads(Path("../tests/gold/olympia_2021.json").read_text(encoding="utf-8"))
MEASURE_OF = {"rate": "rate", "diff": "difference", "hr": "HR", "p": "p", "events": "events"}   # поле эталона → показатель бланка v1


def same_printed(printed, expected):
    """Совпадает ли напечатанное значение с эталоном: числа — как числа (13.0 = 13), строки — без пробелов."""
    if isinstance(expected, str):
        return printed.replace(" ", "") == expected.replace(" ", "")
    try:
        return float(printed) == float(expected)
    except ValueError:
        return False


def found_in_draft(facts, code, field, expected):
    """Есть ли в черновике факт с той же конечной точкой, показателем, группой и значением, что поле эталона."""
    kind = field.split("_")[0]                                   # rate / diff / hr / p / events
    arm = field.split("_")[-1] if kind in ("rate", "events") else None   # olaparib / placebo
    for fact in facts:
        if fact["endpoint"].upper() != code or fact["measure"] != MEASURE_OF[kind]:
            continue
        if arm and arm not in fact["group"].lower():
            continue
        slot = fact[field[field.index("ci_"):]] if "ci_" in field else fact["value"]   # ci_low / ci_high / ci_level или само число
        if slot is not None and same_printed(slot, expected):
            return True
    return False


facts = draft["reading"]["facts"]
missing = []
for code, fields in gold["endpoints"].items():
    for field, expected in fields.items():
        if not found_in_draft(facts, code, field, expected):
            missing.append(f"{code}.{field} = {expected}")
print("черновик Opus: найдено", 22 - len(missing), "из 22 чисел эталона на своём месте · нет:", missing)
print("контроль (HR IDFS 0.85 — такого нет):", found_in_draft(facts, "IDFS", "hr", 0.85))

In [ ]:
import cv2
from app.capture import capture

for name in ["IMG-20230604-WA0010.jpg", "5249372745471040807.jpg", "IMG_20260410_180549.jpg"]:
    slide, outcome = capture(cv2.imread(f"../tests/slides/real/{name}"))
    print(name, "·", outcome, "·", None if slide is None else f"{slide.shape[1]} × {slide.shape[0]}")

In [ ]:
from app.ocr import run_ocr, hint_text

INSTRUCTION_HINT = (
    INSTRUCTION_DRAFT + "\n\n"
    "Ниже — текст этого слайда, распознанный программой OCR, с координатами центра каждой строки "
    "в процентах: x — слева направо, y — сверху вниз. Цифры в OCR точнее, чем на картинке: значения чисел "
    "бери из OCR, а по картинке и координатам определяй, к какой панели, кривой, группе и показателю "
    "относится число. Числа нет в OCR — читай с картинки и добавь пункт в uncertain.\n\n"
)

olympia_ocr = run_ocr(olympia)
print("OCR: строк", len(olympia_ocr["lines"]), "· чисел", len(olympia_ocr["numbers"]))

draft_haiku = read_facts([olympia], "claude-haiku-4-5", INSTRUCTION_HINT + hint_text(olympia_ocr))
RUN_PAID = False
print(save_run(draft_haiku, "draft_olympia_haiku_hint"))
print(f"${draft_haiku['cost_usd']:.4f} · {draft_haiku['seconds']} с · {draft_haiku['stop_reason']} · фактов: {len(draft_haiku['reading']['facts'])}")

missing = []
for code, fields in gold["endpoints"].items():
    for field, expected in fields.items():
        if not found_in_draft(draft_haiku["reading"]["facts"], code, field, expected):
            missing.append(f"{code}.{field} = {expected}")
print("Haiku + OCR: найдено", 22 - len(missing), "из 22 на своём месте · нет:", missing)

In [ ]:
REAL = Path("../tests/slides/real")
SLIDE_SET = {                                  # ключ — короткое имя слайда в наборе; значение — файл фото
    "01_dostarlimab_pfs":      "-5249372745471040796_121.jpg",
    "02_ev302_os":             "5249372745471040807.jpg",
    "03_ras_g12_pfs":          "5253871367231314989.jpg",
    "04_camizestrant_pfs":     "5253871367231314986.jpg",
    "05_bezuclastinib_orr":    "-5249372745471040801_121.jpg",
    "06_atezolizumab_table":   "5278289995770829329.jpg",
    "07_alkove1_baseline":     "-5247120945657356337_121.jpg",
    "08_elisrasib_waterfall":  "-5249372745471040805_121.jpg",
    "09_ev302_orr":            "5249372745471040808.jpg",
    "10_orr_subgroups":        "5254005559189511972.jpg",
    "11_drfi_hr":              "5249372745471040625.jpg",
    "12_oligoprogression":     "5249372745471040750.jpg",
    "13_keynote564_design":    "5249372745471040813.jpg",
    "14_adaura_os_stage":      "IMG-20230604-WA0010.jpg",
    "15_olympia":              sorted(REAL.glob("*.png"))[0].name,   # в имени невидимые пробелы macOS — берём поиском
}

slides = {}
for key, name in SLIDE_SET.items():
    photo = cv2.imread(str(REAL / name))
    if photo is None:                                          # imread не падает, а молча отдаёт None
        print(f"{key:24} ФАЙЛ НЕ ПРОЧИТАН — {name}")
        continue
    slide, outcome = capture(photo)
    if slide is None:
        print(f"{key:24} СЛАЙД НЕ НАЙДЕН — {name}")
        continue
    slides[key] = Image.fromarray(cv2.cvtColor(slide, cv2.COLOR_BGR2RGB))   # BGR OpenCV → RGB Pillow
    print(f"{key:24} {outcome:10} {slides[key].width} × {slides[key].height}")
print("слайдов готово:", len(slides), "из", len(SLIDE_SET))

sheet = Image.new("RGB", (4 * 320, 4 * 190), "white")         # лист-миниатюры: видно, что уходит в модели
for i, img in enumerate(slides.values()):
    thumb = img.copy()
    thumb.thumbnail((310, 180))
    sheet.paste(thumb, ((i % 4) * 320, (i // 4) * 190))
sheet

In [ ]:
from IPython.display import HTML, display
from html import escape
from collections import Counter
import re


def norm(text):
    """Строку → вид для сравнения: заглавные, без пробелов и дефисов; None → пустая строка."""
    return (text or "").upper().replace(" ", "").replace("-", "")


COUNTS = {"n", "events"}                     # «число пациентов» и «число событий» модели путают — это не смысловая ошибка


def same_place(fa, fb):
    """Одинаково ли поняли, что это за число: показатель (n и events — одно) и конечная точка (IRF-pCR ⊃ pCR)."""
    same_measure = fa["measure"] == fb["measure"] or {fa["measure"], fb["measure"]} <= COUNTS
    a, b = norm(fa["endpoint"]), norm(fb["endpoint"])
    return same_measure and (a in b or b in a)


def words(text):
    """Строку → множество слов в нижнем регистре, без знаков: порядок слов не важен."""
    return set(re.findall(r"[a-zа-я0-9+≤≥]+", (text or "").lower()))


def same_group(fa, fb):
    """Похожи ли группы: слова одной записи содержатся в другой, в любом порядке."""
    a, b = words(fa["group"]), words(fb["group"])
    return a <= b or b <= a


def compare_drafts(facts_a, facts_b):
    """Два черновика → строки сверки: совпало / группа разная / место разное / только A / только B."""
    rows = []
    used = set()                                       # факты B, уже нашедшие пару
    for fa in facts_a:
        match = None
        for j, fb in enumerate(facts_b):
            if j in used or norm(fb["value"]) != norm(fa["value"]) or norm(fb["ci_low"]) != norm(fa["ci_low"]):
                continue
            if match is None or same_place(fa, fb):    # из одинаковых чисел предпочитаем то, что на том же месте
                match = j
            if same_place(fa, fb):
                break
        if match is None:
            rows.append(("только A", fa, None))
            continue
        used.add(match)
        fb = facts_b[match]
        if not same_place(fa, fb):
            status = "место разное"
        elif not same_group(fa, fb):
            status = "группа разная"
        else:
            status = "совпало"
        rows.append((status, fa, fb))
    for j, fb in enumerate(facts_b):
        if j not in used:
            rows.append(("только B", None, fb))
    return rows


def seen_by_ocr(value, ocr):
    """Видит ли OCR это значение на слайде: все числа из него — среди чисел OCR; без чисел — поиск в тексте."""
    numbers = re.findall(r"\d+(?:[.,]\d+)?", value)
    if not numbers:
        return norm(value).lower() in ocr["text"]
    return all(float(n.replace(",", ".")) in ocr["numbers"] for n in numbers)


COLORS = {"совпало": "#e6f4ea", "группа разная": "#fff4cc", "место разное": "#ffe0b2",
          "только A": "#fde2e1", "только B": "#fde2e1"}


def fact_text(f):
    """Факт → короткая строка для таблицы сверки; escape — чтобы «<0.001» не приняли за разметку."""
    if f is None:
        return "—"
    ci = f" ({f['ci_level']}% ДИ {f['ci_low']}–{f['ci_high']})" if f["ci_low"] else ""
    when = f" · {f['timepoint']}" if f["timepoint"] else ""
    return escape(f"{f['measure']} · {f['endpoint']} · {f['group']} · ") + f"<b>{escape(f['value'])}</b>" + escape(f"{ci}{when}")


def show_check(img, rows, ocr, name_a, name_b):
    """Слайд + таблица сверки: цвет — согласие черновиков, «OCR» — видит ли программа число на слайде."""
    display(img)
    html = [f"<table style='color:#111'><tr><th>№</th><th>статус</th><th>OCR</th><th>{name_a}</th><th>{name_b}</th></tr>"]
    for i, (status, fa, fb) in enumerate(rows):
        seen = "да" if seen_by_ocr((fa or fb)["value"], ocr) else "<b>НЕТ</b>"
        html.append(f"<tr style='background:{COLORS[status]}'><td>{i}</td><td>{status}</td><td>{seen}</td>"
                    f"<td>{fact_text(fa)}</td><td>{fact_text(fb)}</td></tr>")
    html.append("</table>")
    display(HTML("".join(html)))
    print(dict(Counter(status for status, fa, fb in rows)))

In [ ]:
from html import unescape
from datetime import date

GOLD_DIR = Path("../data/gold_set")          # эталон набора — не в git: расшифровка чужих слайдов (правило 36а)


def pick(rows, number, side="A", fix=None):
    """Строка таблицы сверки → ключевое утверждение: факт черновика A или B, при необходимости с правкой полей."""
    status, fa, fb = rows[number]
    chosen = fa if side == "A" else fb
    if chosen is None:
        raise ValueError(f"в строке {number} нет черновика {side} — возьми другую сторону")
    claim = dict(chosen)                     # копия: черновик не трогаем
    if fix:
        claim.update(fix)                    # например {"value": "79.6", "source": "Geyer 2022, Fig. 2C"} — решение № 35
    return claim


def save_gold(key, study, key_claims, slide_problems, questions, overwrite=False):
    """Эталон слайда (решение № 36) → data/gold_set/<ключ>.json; вернуть путь. Существующий — только с overwrite=True."""
    if (GOLD_DIR / f"{key}.json").exists() and not overwrite:
        raise FileExistsError(f"эталон {key} уже сохранён — перезапись только с overwrite=True")
    if not 1 <= len(key_claims) <= 6 and study.get("type") != "design":
        print(f"внимание: ключевых утверждений {len(key_claims)} — по решению № 36 их 3–6")
    gold = {
        "slide": key,
        "study": study,
        "key_claims": key_claims,
        "slide_problems": slide_problems,
        "questions": questions,
        "verified_by": "Иван",
        "date": date.today().isoformat(),
    }
    GOLD_DIR.mkdir(parents=True, exist_ok=True)
    path = GOLD_DIR / f"{key}.json"
    path.write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def show_gold(path):
    """Показать сохранённый эталон коротко — проверить глазами, что записалось."""
    gold = json.loads(Path(path).read_text(encoding="utf-8"))
    print("исследование:", gold["study"])
    for claim in gold["key_claims"]:
        text = unescape(fact_text(claim).replace("<b>", "").replace("</b>", ""))
        print("  •", text, f"[{claim['source']}]" if claim.get("source") else "")
    print("проблемы слайда:", gold["slide_problems"])
    print("вопросы:", gold["questions"])

In [ ]:
QUICK = ["04_camizestrant_pfs", "02_ev302_os", "05_bezuclastinib_orr"]   # 3 слайда обычными вызовами, пока идёт пакет
quick_drafts = {}
quick_cost = 0.0

for key in QUICK:
    for short, model in DRAFT_MODELS.items():
        try:
            run = read_facts([slides[key]], model, INSTRUCTION_DRAFT)
        except Exception as error:
            print(f"{key:22} {short:6} ошибка: {type(error).__name__}")
            continue
        quick_cost += run["cost_usd"]
        save_run(run, f"draft_{key}_{short}")
        quick_drafts.setdefault(key, {})[short] = run["reading"]
        print(f"{key:22} {short:6} ${run['cost_usd']:.4f} · {run['seconds']} с · {run['stop_reason']} · фактов: {len(run['reading']['facts'])}")
RUN_PAID = False
print(f"расход: ${quick_cost:.4f}")

In [ ]:
def load_draft(key, model):
    """Последний черновик слайда с диска: data/runs/…_draft_<слайд>_<модель>.json."""
    paths = sorted(Path("../data/runs").glob(f"*_draft_{key}_{model}.json"))
    return json.loads(paths[-1].read_text(encoding="utf-8"))["reading"]


drafts = {}
for key in SLIDE_SET:
    drafts[key] = {"opus": load_draft(key, "opus"), "sonnet": load_draft(key, "sonnet")}


def rows_for(key):
    """Таблица сверки Opus против Sonnet для одного слайда."""
    return compare_drafts(drafts[key]["opus"]["facts"], drafts[key]["sonnet"]["facts"])


for key in SLIDE_SET:
    print(f"{key:24}", dict(Counter(status for status, fa, fb in rows_for(key))))

In [ ]:
r01 = rows_for("01_dostarlimab_pfs")
display(slides["01_dostarlimab_pfs"])
path = save_gold("01_dostarlimab_pfs",
    study={"name": "RUBY", "nct": "NCT03981796",
           "source": "ClinicalTrials.gov; Powell MA et al., Gynecol Oncol 2026 (найдено поиском); на слайде — SGO 2026"},
    key_claims=[pick(r01, 0), pick(r01, 4), pick(r01, 7), pick(r01, 9)],   # HR 0.30; 4-летняя ВБП 57.9 и 15.7; медиана плацебо 7.7
    slide_problems=[],
    questions=[])
show_gold(path)

In [ ]:
r03 = rows_for("03_ras_g12_pfs")
display(slides["03_ras_g12_pfs"])
path = save_gold("03_ras_g12_pfs",
    study={"name": None, "nct": None,
           "source": "не опознано: вероятно, дараксонрасиб (RMC-6236-001) — НЕ ПРОВЕРЕНО"},
    key_claims=[pick(r03, 0), pick(r03, 1), pick(r03, 2), pick(r03, 3)],   # n 33 и 38; медианы ВБП 8.3 и 8.3
    slide_problems=["исследование и препарат на слайде не указаны"],
    questions=[])
show_gold(path)

In [ ]:
r06 = rows_for("06_atezolizumab_table")
display(slides["06_atezolizumab_table"])
path = save_gold("06_atezolizumab_table",
    study={"name": "IMpower030 (название — по поиску)", "nct": "NCT03456063",
           "source": "ClinicalTrials.gov (неоадъювантный атезолизумаб, стадии II–IIIB); WCLC 2026 — тезис не найден"},
    key_claims=[pick(r06, 24), pick(r06, 25), pick(r06, 5), pick(r06, 7), pick(r06, 20), pick(r06, 21)],
    slide_problems=["НЕ ПРОВЕРЕНО: в пресс-релизе IASLC медиана EFS плацебо 34.9 мес, на слайде 34.6"],
    questions=[])
show_gold(path)

In [ ]:
r07 = rows_for("07_alkove1_baseline")
display(slides["07_alkove1_baseline"])
path = save_gold("07_alkove1_baseline",
    study={"name": "ALKOVE-1", "nct": "NCT05384626", "source": "ClinicalTrials.gov; Lin J, ASCO 2026 abstr 8503 (найдено поиском)"},
    key_claims=[pick(r07, 0),                                                          # N = 253 (после ALK-ИТК)
                pick(r07, 1, fix={"ci_level": None, "ci_low": None, "ci_high": None,
                                  "note": "возраст: разброс 24–83, не ДИ"}),           # медиана возраста 56
                pick(r07, 17), pick(r07, 19), pick(r07, 21)],                          # мтс в ЦНС 40 %; вторичные мутации ALK 36 %; G1202R 19 %
    slide_problems=[],
    questions=[])
show_gold(path)

In [ ]:
r08 = rows_for("08_elisrasib_waterfall")
display(slides["08_elisrasib_waterfall"])
path = save_gold("08_elisrasib_waterfall",
    study={"name": "элисрасиб, 1-я линия НМРЛ (название исследования на слайде нет)", "nct": None,
           "source": "Lu S, ASCO 2026 abstr 8511 (найдено поиском)"},
    key_claims=[pick(r08, 0), pick(r08, 15), pick(r08, 26)],   # ЧОО 78.0; подтверждённая ЧОО 68.3; контроль болезни 95.1
    slide_problems=["крупно — ЧОО 78.0 % с неподтверждёнными ответами; подтверждённая ЧОО — 68.3 %"],
    questions=[])
show_gold(path)

In [ ]:
r09 = rows_for("09_ev302_orr")
display(slides["09_ev302_orr"])
path = save_gold("09_ev302_orr",
    study={"name": "EV-302", "nct": "NCT04223856",
           "source": "Powles, Ann Oncol 2025 (найдено поиском); на слайде — Gupta ASCO 2025 abstr 4502, ASCO GU 2026 abstr 746"},
    key_claims=[pick(r09, 3), pick(r09, 9), pick(r09, 5), pick(r09, 11), pick(r09, 27)],   # ЧОО 67.5/44.2; ПО 30.4/14.5; 66.2 % ПО — после ЧО
    slide_problems=["в одном докладе: ответ — срез 08.08.2024 (BICR прекращён), ОВ (слайд 02) — срез 06.10.2025"],
    questions=[])
show_gold(path)

In [ ]:
r10 = rows_for("10_orr_subgroups")
display(slides["10_orr_subgroups"])
path = save_gold("10_orr_subgroups",
    study={"name": "TRITON", "nct": "NCT06008093",
           "source": "ClinicalTrials.gov: рандомизированная фаза IIb, открытая, 103 пациента; Skoulidis, ASCO 2026 abstr 8515 (найдено поиском)"},
    key_claims=[pick(r10, 0), pick(r10, 2), pick(r10, 8), pick(r10, 10), pick(r10, 12), pick(r10, 14)],   # ЧОО STK11m, KRASm, «только KRASm»
    slide_problems=[],
    questions=[])
show_gold(path)

In [ ]:
r11 = rows_for("11_drfi_hr")
display(slides["11_drfi_hr"])
path = save_gold("11_drfi_hr",
    study={"name": "OPTIMA", "nct": None, "source": "Stein R, ASCO 2026 abstr 500 (найдено поиском)"},
    key_claims=[pick(r11, 0), pick(r11, 3), pick(r11, 4), pick(r11, 6)],   # HR 1.04 (90% ДИ); 5-летний DRFI 94.1 и 93.3; HR 1.17 при ROR ≤60
    slide_problems=["НЕ ПРОВЕРЕНО: чисел DRFI со слайда нет в тезисе ASCO 2026 (там только IBCFS)"],
    questions=[])
show_gold(path)

In [ ]:
r12 = rows_for("12_oligoprogression")
display(slides["12_oligoprogression"])
path = save_gold("12_oligoprogression",
    study={"name": "обзор: Gomez 2019; Iyengar 2018; Weickhardt 2012", "nct": None, "source": "по ссылкам на слайде"},
    key_claims=[pick(r12, 0), pick(r12, 1), pick(r12, 2), pick(r12, 3), pick(r12, 4), pick(r12, 5)],
    slide_problems=["НЕ ПРОВЕРЕНО: «6 to 9 m» — у Weickhardt 2012 PFS2 6.2 мес и «более 6 мес»"],
    questions=[])
show_gold(path)

In [ ]:
display(slides["13_keynote564_design"])
path = save_gold("13_keynote564_design",
    study={"name": "KEYNOTE-564", "nct": "NCT03142334", "source": "номер на слайде; ClinicalTrials.gov", "type": "design"},
    key_claims=[],                                   # дизайн: верный ответ модели — пустой список (Opus — 0, Sonnet — 8)
    slide_problems=[],
    questions=[])
show_gold(path)

In [ ]:
r14 = rows_for("14_adaura_os_stage")
display(slides["14_adaura_os_stage"])
path = save_gold("14_adaura_os_stage",
    study={"name": "ADAURA", "nct": "NCT02511106",
           "source": "ClinicalTrials.gov; Tsuboi M et al., NEJM 2023, doi:10.1056/NEJMoa2304594 (найдено поиском)"},
    key_claims=[pick(r14, 6), pick(r14, 7), pick(r14, 8), pick(r14, 4), pick(r14, 5)],   # HR ОВ по стадиям IB, II, IIIA; 5-летняя ОВ IIIA 85 и 67
    slide_problems=[],
    questions=[])
show_gold(path)

In [ ]:
from pydantic import ValidationError

gold_set = {}
for path in sorted(GOLD_DIR.glob("*.json")):
    gold_set[path.stem] = json.loads(path.read_text(encoding="utf-8"))

batch_raw = json.loads(Path("../data/runs/batch_drafts_results.json").read_text(encoding="utf-8"))


def batch_draft(key, model):
    """Второй прогон — ответ отменённого пакета (27 из 30 успели); нет или оборван — None."""
    item = batch_raw.get(f"{key}__{model}")
    if not item or "text" not in item:
        return None
    try:
        return SlideFacts.model_validate_json(item["text"]).model_dump()
    except ValidationError:
        return None                          # ответ оборван: закончилось место (у Sonnet на OlympiA)


def has_value(fact, value):
    """Значение то же: совпадает целиком или стоит внутри записи вида «32 (78.0%)»."""
    return norm(fact["value"]) == norm(value) or value in re.findall(r"[<>≤≥]?\d+(?:[.,]\d+)?", fact["value"])


def score_claims(reading, gold):
    """Ключевые утверждения эталона против чтения модели → исход по каждому (решение № 36)."""
    doubts = " ".join(reading["uncertain"]).lower()
    outcomes = []
    for claim in gold["key_claims"]:
        same_value = [f for f in reading["facts"] if has_value(f, claim["value"])]
        same_spot = [f for f in reading["facts"] if same_place(claim, f) and same_group(claim, f)]
        if any(f in same_spot for f in same_value):
            outcome = "верно"
        elif same_value:
            outcome = "место другое"             # число то же, отнесено не туда — кандидат в принципиальную ошибку
        elif same_spot:
            outcome = "число другое"             # место то же, цифра другая
        else:
            outcome = "нет"
        if outcome != "верно" and claim["value"].lower() in doubts:
            outcome += " (с пометкой)"
        outcomes.append((claim, outcome))
    return outcomes


def score_reader(get_reading):
    """Прогнать одного «читателя» по всему набору: счёт исходов, прошедшие слайды, список расхождений."""
    counts, passed, slides_scored, misses = Counter(), 0, 0, []
    for key, gold in gold_set.items():
        reading = get_reading(key)
        if reading is None:
            continue
        slides_scored += 1
        outcomes = score_claims(reading, gold)
        counts.update(outcome for claim, outcome in outcomes)
        if not any(outcome == "место другое" for claim, outcome in outcomes):
            passed += 1
        misses += [(key, claim["value"], outcome) for claim, outcome in outcomes if outcome != "верно"]
    return counts, passed, slides_scored, misses


readers = {
    "Opus, прогон 1":   lambda key: load_draft(key, "opus"),
    "Opus, прогон 2":   lambda key: batch_draft(key, "opus"),
    "Sonnet, прогон 1": lambda key: load_draft(key, "sonnet"),
    "Sonnet, прогон 2": lambda key: batch_draft(key, "sonnet"),
}
for name, get_reading in readers.items():
    counts, passed, n, misses = score_reader(get_reading)
    print(f"{name:18} слайдов {n:2} · прошли {passed:2} · {dict(counts)}")
    for miss in misses:
        print("     ", miss)

In [ ]:
haiku_cost = 0.0
RUN_PAID = True                                          # сознательно: Haiku + OCR × 15 слайдов, ≈ $0,2
for key in SLIDE_SET:
    try:
        run = read_facts([slides[key]], "claude-haiku-4-5", INSTRUCTION_HINT + hint_text(run_ocr(slides[key])))
    except Exception as error:
        print(f"{key:24} ошибка: {type(error).__name__}")
        continue
    haiku_cost += run["cost_usd"]
    save_run(run, f"draft_{key}_haiku")
    print(f"{key:24} ${run['cost_usd']:.4f} · {run['seconds']} с · фактов: {len(run['reading']['facts'])}")
RUN_PAID = False
print(f"расход: ${haiku_cost:.4f}")

counts, passed, n, misses = score_reader(lambda key: load_draft(key, "haiku"))
print(f"Haiku + OCR        слайдов {n:2} · прошли {passed:2} · {dict(counts)}")
for miss in misses:
    print("     ", miss)